# Phase 2: Autograd — Automatic Differentiation

## What you'll learn
- How PyTorch tracks operations to compute gradients
- `requires_grad`, `.backward()`, `.grad`
- The computation graph
- Detaching and disabling gradients
- Gradient accumulation
- Custom autograd functions

---

## Why Autograd?

Deep learning = finding parameters that minimize a loss function.  
To minimize, we need **gradients** (derivatives).  
Autograd computes these **automatically** by recording every operation on tensors.

In [ ]:
import torch

## 2.1 requires_grad — Telling PyTorch to Track Gradients

Set `requires_grad=True` on any tensor you want gradients for (typically model weights).

In [ ]:
# Create a tensor that tracks gradients
x = torch.tensor(3.0, requires_grad=True)
print(f"x = {x}")
print(f"requires_grad = {x.requires_grad}")

# Any operation on x creates a computation graph
y = x ** 2 + 2 * x + 1   # y = x² + 2x + 1
print(f"y = {y}")
print(f"y.grad_fn = {y.grad_fn}")  # shows the last operation

## 2.2 .backward() — Computing Gradients

Call `.backward()` on a scalar output to compute all gradients via the chain rule (backpropagation).

For `y = x² + 2x + 1`, the derivative is `dy/dx = 2x + 2`.  
At `x = 3`: `dy/dx = 2(3) + 2 = 8`.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x + 1

# Compute gradients
y.backward()

# The gradient is stored in x.grad
print(f"dy/dx at x=3: {x.grad}")  # 8.0 ✓

## 2.3 Multi-variable Gradients

Autograd handles functions of multiple variables — it computes partial derivatives for each.

In [ ]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

# z = a²b + b³
z = a**2 * b + b**3

z.backward()

# dz/da = 2ab = 2*2*3 = 12
print(f"dz/da = {a.grad}")  # 12.0

# dz/db = a² + 3b² = 4 + 27 = 31
print(f"dz/db = {b.grad}")  # 31.0

## 2.4 Gradients with Vectors

When working with vector outputs, you need to reduce to a scalar first (e.g., `.sum()` or `.mean()`).

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# y = x² (element-wise)
y = x ** 2

# Can't call y.backward() directly — y is not scalar
# Reduce to scalar first
loss = y.sum()  # or y.mean()
loss.backward()

# d(sum(x²))/dx = 2x
print(f"Gradients: {x.grad}")  # [2, 4, 6]

## 2.5 The Computation Graph

PyTorch builds a **dynamic computation graph** (DAG) as you run operations.

```
x (leaf) → x² → +2x → +1 → y (output)
```

Key points:
- **Leaf tensors**: created by the user (e.g., `x`)
- **Non-leaf tensors**: results of operations (e.g., `y`)
- The graph is **destroyed after `.backward()`** (one-time use)
- Use `retain_graph=True` if you need to call `.backward()` again

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 3
z = y ** 2

print(f"x is leaf: {x.is_leaf}")   # True
print(f"y is leaf: {y.is_leaf}")   # False
print(f"y.grad_fn: {y.grad_fn}")   # MulBackward0
print(f"z.grad_fn: {z.grad_fn}")   # PowBackward0

z.backward()
print(f"\ndz/dx = {x.grad}")  # dz/dx = 2*(3x)*3 = 18x = 36

## 2.6 Gradient Accumulation & Zeroing

**Critical concept**: PyTorch **accumulates** gradients by default. You must zero them before each backward pass.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

# First backward
y1 = x ** 2
y1.backward()
print(f"After 1st backward: {x.grad}")  # 4.0

# Second backward WITHOUT zeroing — gradients accumulate!
y2 = x ** 2
y2.backward()
print(f"After 2nd backward (accumulated): {x.grad}")  # 8.0 (4+4)

# Correct approach: zero gradients first
x.grad.zero_()
y3 = x ** 2
y3.backward()
print(f"After zeroing + backward: {x.grad}")  # 4.0 ✓

## 2.7 Disabling Gradient Tracking

During inference or evaluation, you don't need gradients. Disabling them saves memory and speeds things up.

Three ways to do it:

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

# Method 1: torch.no_grad() context manager
with torch.no_grad():
    y = x * 2
    print(f"no_grad — y.requires_grad: {y.requires_grad}")  # False

# Method 2: .detach() — creates a new tensor without grad
z = x.detach()
print(f"detach — z.requires_grad: {z.requires_grad}")  # False

# Method 3: set requires_grad directly
a = torch.tensor(5.0, requires_grad=True)
a.requires_grad_(False)
print(f"requires_grad_(False): {a.requires_grad}")  # False

## 2.8 Practical Example: Manual Linear Regression

Let's use autograd to train a simple linear model `y = wx + b` from scratch.

In [ ]:
# Generate synthetic data: y = 3x + 1 + noise
torch.manual_seed(42)
X = torch.rand(100, 1) * 10
y_true = 3 * X + 1 + torch.randn(100, 1) * 0.5

# Initialize learnable parameters
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.01

# Training loop
for epoch in range(100):
    # Forward pass
    y_pred = w * X + b
    loss = ((y_pred - y_true) ** 2).mean()  # MSE

    # Backward pass
    loss.backward()

    # Update weights (no_grad because we don't want to track this)
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

    # Zero gradients
    w.grad.zero_()
    b.grad.zero_()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: loss={loss.item():.4f}, w={w.item():.4f}, b={b.item():.4f}")

print(f"\nLearned: y = {w.item():.2f}x + {b.item():.2f}")
print(f"True:    y = 3.00x + 1.00")

## 2.9 Custom Autograd Function (Advanced)

You can define custom forward/backward passes by subclassing `torch.autograd.Function`.

In [ ]:
class MyReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        # ctx is a context object to save info for backward
        ctx.save_for_backward(x)
        return x.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        grad = grad_output.clone()
        grad[x < 0] = 0  # gradient is 0 where x < 0
        return grad

# Usage
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0], requires_grad=True)
y = MyReLU.apply(x)
y.sum().backward()

print(f"Input:    {x.data}")
print(f"Output:   {y.data}")
print(f"Gradient: {x.grad}")  # [0, 0, 0, 1, 1]

## ✅ Phase 2 Summary

You now understand:
- `requires_grad=True` tells PyTorch to track operations
- `.backward()` computes gradients via backpropagation
- `.grad` stores the computed gradient on leaf tensors
- Gradients **accumulate** — always zero them before each step
- Use `torch.no_grad()` or `.detach()` to stop tracking
- The computation graph is dynamic and rebuilt each forward pass

**Next → Phase 3: Neural Network Building Blocks (nn.Module)**